# Simple RNN text classification

Train a small RNN to label short movie comments as **positive** or **negative**. This is a learning example with a deliberately tiny dataset.

In [6]:
import pandas as pd

# Load IMDb dataset
file_path = "/content/IMDB Dataset.csv"

df = pd.read_csv(file_path)

# Show original dataset size
print("Original dataset shape:", df.shape)

# Take a smaller random sample for faster training
df_small = df.sample(n=2500, random_state=42).reset_index(drop=True)

# Show subsampled dataset size
print("Subsampled dataset shape:", df_small.shape)

# Display first 5 rows
df_small.head()

Original dataset shape: (50000, 2)
Subsampled dataset shape: (2500, 2)


,review,sentiment
0,I really liked this Summerslam due to the look...,positive
1,Not many television shows appeal to quite as m...,positive
2,The film quickly gets to a major chase scene w...,negative
3,Jane Austen would definitely approve of this o...,positive
4,Expectations were somewhat high for me when I ...,negative


In [7]:
# Convert sentiment labels to numeric values
# positive = 1
# negative = 0

df_small["label"] = df_small["sentiment"].map({
    "positive": 1,
    "negative": 0
})

# Check the result
print(df_small[["sentiment", "label"]].head())

# Check label distribution
print("\nLabel distribution:")
print(df_small["label"].value_counts())

  sentiment  label
0  positive      1
1  positive      1
2  negative      0
3  positive      1
4  negative      0

Label distribution:
label
1    1261
0    1239
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

# Features (movie reviews) and labels
X = df_small["review"]
y = df_small["label"]

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Check sizes
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 2000
Testing samples: 500


In [9]:
# Install tiktoken if it is not already installed
!pip install tiktoken -q

import tiktoken

# Load cl100k_base tokenizer
encoding = tiktoken.get_encoding("cl100k_base")

# Convert each review into a list of token IDs
X_train_tokens = X_train.apply(lambda review: encoding.encode(str(review)))
X_test_tokens = X_test.apply(lambda review: encoding.encode(str(review)))

# Display examples
print("First review:")
print(X_train.iloc[0][:200], "...")

print("\nToken IDs:")
print(X_train_tokens.iloc[0][:30])

print("\nNumber of tokens:", len(X_train_tokens.iloc[0]))

First review:
Valley Girl is the definitive 1980's movie with catch phrases filtered throughout this wonderfully acted movie. The characters are so convincing that you forget it is a movie and not a video of an act ...

Token IDs:
[2257, 3258, 11617, 374, 279, 45813, 220, 3753, 15, 596, 5818, 449, 2339, 32847, 18797, 6957, 420, 61085, 31532, 5818, 13, 578, 5885, 527, 779, 40661, 430, 499, 10894, 433]

Number of tokens: 94


In [10]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Maximum sequence length
MAX_LEN = 200

# Pad or truncate training sequences
X_train_padded = pad_sequences(
    X_train_tokens,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
    value=0
)

# Pad or truncate testing sequences
X_test_padded = pad_sequences(
    X_test_tokens,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
    value=0
)

# Check shapes
print("Training data shape:", X_train_padded.shape)
print("Testing data shape:", X_test_padded.shape)

# Display one padded sequence
print("\nFirst padded sequence:")
print(X_train_padded[0])

Training data shape: (2000, 200)
Testing data shape: (500, 200)

First padded sequence:
[ 2257  3258 11617   374   279 45813   220  3753    15   596  5818   449
  2339 32847 18797  6957   420 61085 31532  5818    13   578  5885   527
   779 40661   430   499 10894   433   374   264  5818   323   539   264
  2835   315   459  5150   330  1316  3502 10826 26928     1   315   904
  1579  2978    11  7427    13  1115 29447   374   311   279   220  3753
    15   596  1148   279 36470   426  3265  6007  4101   374   311   279
   220  4468    15   596    13  1442   499  1541   956  1093   433    11
  1664  1243   330    38   351   757   449   264 46605  1210     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0

In [11]:
import torch

# Convert padded token sequences into PyTorch tensors
X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)

# Convert labels into PyTorch tensors
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

# Check tensor shapes
print("X_train tensor shape:", X_train_tensor.shape)
print("y_train tensor shape:", y_train_tensor.shape)

print("X_test tensor shape:", X_test_tensor.shape)
print("y_test tensor shape:", y_test_tensor.shape)

# Check tensor data types
print("\nData types:")
print("X_train:", X_train_tensor.dtype)
print("y_train:", y_train_tensor.dtype)

X_train tensor shape: torch.Size([2000, 200])
y_train tensor shape: torch.Size([2000])
X_test tensor shape: torch.Size([500, 200])
y_test tensor shape: torch.Size([500])

Data types:
X_train: torch.int64
y_train: torch.int64


In [12]:
import torch
import torch.nn as nn

# Model parameters
VOCAB_SIZE = encoding.n_vocab
EMBED_DIM = 64
HIDDEN_DIM = 128
NUM_CLASSES = 2

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super(SentimentRNN, self).__init__()

        # 1. Embedding layer
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        # 2. RNN layer
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # 3. Linear output layer
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):

        # Token IDs → embedding vectors
        x = self.embedding(x)

        # Embeddings → RNN
        output, hidden = self.rnn(x)

        # Take the final hidden state
        x = hidden[-1]

        # RNN output → class scores
        x = self.fc(x)

        return x


# Create model
model = SentimentRNN(
    VOCAB_SIZE,
    EMBED_DIM,
    HIDDEN_DIM,
    NUM_CLASSES
)

print(model)

SentimentRNN(
  (embedding): Embedding(100277, 64, padding_idx=0)
  (rnn): RNN(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


In [13]:
from torch.utils.data import TensorDataset, DataLoader

# Create training dataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

# Create batches
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Number of training epochs
EPOCHS = 5

# Training loop
for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(batch_X)

        # Calculate loss
        loss = criterion(outputs, batch_y)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        total_loss += loss.item()

    # Average loss
    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch + 1}/{EPOCHS}], Loss: {avg_loss:.4f}")

Epoch [1/5], Loss: 0.7072
Epoch [2/5], Loss: 0.6682
Epoch [3/5], Loss: 0.6411
Epoch [4/5], Loss: 0.5986
Epoch [5/5], Loss: 0.6276


In [14]:
# Put the model in evaluation mode
model.eval()

# Disable gradient calculation
with torch.no_grad():

    # Get predictions for the test set
    outputs = model(X_test_tensor)

    # Select the class with the highest score
    predictions = torch.argmax(outputs, dim=1)

    # Calculate number of correct predictions
    correct = (predictions == y_test_tensor).sum().item()

    # Calculate accuracy
    accuracy = correct / len(y_test_tensor)

print(f"Correct predictions: {correct}/{len(y_test_tensor)}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Correct predictions: 236/500
Test Accuracy: 47.20%


In [24]:
review = " movie was bad"

x = torch.tensor([encoding.encode(review)[:MAX_LEN]], dtype=torch.long)
x = torch.nn.functional.pad(x, (0, MAX_LEN - x.shape[1]))

model.eval()
with torch.no_grad():
    prediction = model(x).argmax(1).item()

print("Positive" if prediction == 1 else "Negative")

Negative
